# SubaRAG compact-runtime screen
Run in a **fresh free Colab CPU runtime**. This downloads the already-built private pilot/indexes, exports the pinned E5 model to an experimental 8-bit ONNX format, then measures reference and candidate in separate CPU processes. It does not rebuild passage embeddings, call Gemini/ElevenLabs, upload artifacts, or change Render.

Use `SubaRAG-compact-source.zip` when prompted, not a previous source ZIP. Enter only the dataset-scoped read token in the hidden prompt. No write token is needed. A passing screen is not yet proof of safe Render deployment: Colab CPU speed and process RSS differ from Render's constrained container.

In [ ]:
from google.colab import files
from pathlib import Path
from getpass import getpass
import hashlib, io, json, os, subprocess, sys, tempfile, zipfile

def run_logged(arguments, *, env=None):
    with subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    if code != 0:
        raise RuntimeError(f'Process stopped with exit code {code}. Keep the runtime and share the last error lines without secrets.')

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one compact-runtime source ZIP. Renamed copies are accepted.')
archive_bytes = next(iter(uploaded.values()))
source_root = Path(tempfile.mkdtemp(prefix='subarag-compact-source-', dir='/content'))
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    archive.getinfo('SubaRAG/backend/src/subarag/experiments/compact_embedding.py')
    for member in archive.infolist():
        destination = (source_root / member.filename).resolve()
        if not destination.is_relative_to(source_root) or (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('Unsafe source ZIP path or symlink.')
    archive.extractall(source_root)
backend = source_root / 'SubaRAG/backend'
run_logged([sys.executable, '-m', 'pip', 'install', f'{backend}[rag,compact]', 'onnx>=1.17,<2'])
run_logged([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps', str(backend)])
child_env = dict(os.environ, PYTHONPATH=str(backend / 'src'), OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', TOKENIZERS_PARALLELISM='false')
child_env.pop('HF_TOKEN', None)
WORK = Path(tempfile.mkdtemp(prefix='subarag-compact-', dir='/content'))
def stage(name, *, env=None):
    run_logged([sys.executable, '-u', '-m', 'subarag.experiments.compact_embedding', name, '--work', str(WORK)], env=child_env if env is None else env)
print('Work folder:', WORK)
print('Source SHA-256:', hashlib.sha256(archive_bytes).hexdigest())


In [ ]:
read_env = dict(child_env)
read_env['HF_TOKEN'] = getpass('HF read token for private SubaRAG-data (hidden): ').strip()
try:
    stage('prepare', env=read_env)
finally:
    del read_env


In [ ]:
stage('export')


In [ ]:
stage('reference')
stage('candidate')
stage('report')
report = json.loads((WORK / 'compact-report.json').read_text())
report['source_archive_sha256'] = hashlib.sha256(archive_bytes).hexdigest()
(WORK / 'compact-report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
files.download(str(WORK / 'compact-report.json'))
print('Share compact-report.json and keep this runtime open. Do not upload the model or change Render settings yet.')
